# Colab Runner — Hackachusetts
Code lives on GitHub; this notebook only **runs** it (CLAUDE.md §5).

1. Runtime → Change runtime type → **T4 GPU** (only needed for foundation models / neural nets).
2. Run cells 1–4 every session.
3. Cell 5 (downloads) only once, by **P1**, when making the processed files.
4. Cell 6: run **one** job per session. Always `--quick` first.

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Get the latest code
# (If the repo is private, use https://<TOKEN>@github.com/... with a GitHub personal access token.)
!git clone https://github.com/nealp/foundationalModelsForWearableTech.git /content/repo || (cd /content/repo && git pull)
%cd /content/repo

In [ ]:
# 3. Install core packages
!pip install -q -r requirements.txt

In [ ]:
# 3b. OPTIONAL: install ONE foundation model / neural package for this session's job.
# Uncomment only the line you need.
# !pip install -q chronos-forecasting
# !pip install -q timesfm
# !pip install -q momentfm
# !pip install -q neuralforecast

In [ ]:
# 4. Point the code at Drive (all paths come from src/config.py)
import os
os.environ["DATA_DIR"] = "/content/drive/MyDrive/hackachusetts"
os.environ["RAW_DIR"] = "/content/raw"   # raw data stays on temporary disk, NEVER on Drive
!python -c "from src import config; config.ensure_dirs(); print('DATA_DIR =', config.DATA_DIR)"

## 5. Downloads (P1 only, once)
Raw data goes to `/content/raw` and disappears when the session ends. That's on purpose.

In [ ]:
# 5a. WESAD (~2 GB). Paste the official download link from the WESAD page below.
WESAD_URL = ""  # <-- human pastes link here
assert WESAD_URL, 'Paste the WESAD link first'
!mkdir -p /content/raw/wesad
!wget -q --show-progress -O /content/raw/wesad/WESAD.zip "$WESAD_URL"
!unzip -q -o /content/raw/wesad/WESAD.zip -d /content/raw/wesad
!ls /content/raw/wesad/WESAD | head

In [ ]:
# 5b. Sleep-EDF sleep-cassette, first 20 people (use range(10) if short on time)
from mne.datasets.sleep_physionet import age
files = age.fetch_data(subjects=list(range(20)), recording=[1, 2], path='/content/raw/sleepedf', on_missing='warn')
print(len(files), 'recordings downloaded')

## 6. Run ONE job
Always run with `--quick` first (2–3 subjects, 1 fold). Only run the full job if quick works.

In [ ]:
!python -m src.run --task hr --model naive --quick
# !python -m src.run --task hr --model naive